### 📥 Instalações

### ⚙️ Configurações

In [ ]:
CONFIG = {
  #================================#
  #=== CONFIGURAÇÕES DO DATASET ===#
  #================================#
  "dataset": "../../data/Transporte.csv",
  "freq": "h",

  #===============================#
  #=== CONJUNTO DE TREINAMENTO ===#
  #===============================#
  "train_start": "2018-01-10 03:00:00",
  "train_end": "2018-03-10 02:00:00",

  #===============================#
  #=== CONFIGURAÇÕES DO MODELO ===#
  #===============================#
  "model": "gemma-3-27b-it",
  "api_key": "lm-studio",
  "base_url": "http://192.168.68.113:1234/v1",
  "temperatures": [0.7],

  #=========================#
  #=== PREÇO / 1M TOKENS ===#
  #=========================#
  "input_price": 0,
  "output_price": 0,

  #===============================#
  #=== CONFIGURAÇÕES DO PROMPT ===#
  #===============================#
  "forecast_horizon": 24,
  "formats": ["plain", "csv"],
  "types": ["numeric", "textual"],
  "sampling": ["uniform"],
  "top_k": [3],

  #======================================#
  #=== CONFIGURAÇÕES DOS EXPERIMENTOS ===#
  #======================================#
  "runs_per_prompt": 5,
  "max_retries": 10,
  "delay": 0,

  #==============#
  #=== OUTPUT ===#
  #==============#
  "folder": "../../experiments/Transporte_Gemma_27b"
}

# 📦 1. Importações

In [ ]:
import llm4series as ls
import pandas as pd
import plotly.express as px
from scipy.stats import sem
import time
import os

import logging
logging.getLogger("apscheduler").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)
logging.getLogger("requests").setLevel(logging.CRITICAL)

import warnings
warnings.simplefilter("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# === CarbonTracker ===
import io
import re
from contextlib import redirect_stdout
from carbontracker.tracker import CarbonTracker as Tracker

# === Eco2AI ===
import eco2ai
eco2ai_tracker = eco2ai.Tracker(project_name="eco2ai", file_name="eco2ai_emissions.csv", ignore_warnings=True)

# 🌱 2. Pegada de carbono

In [ ]:
class CarbonTracker:
  def __init__(self, epochs=1, verbose=1):
    self.epochs = epochs
    self.verbose = verbose
    self.buffer = io.StringIO()
    self.tracker = None
    self.co2_g = None
    self.energy_kwh = None

  def __enter__(self):
    self._stdout_ctx = redirect_stdout(self.buffer)
    self._stdout_ctx.__enter__()
    self.tracker = Tracker(epochs=self.epochs, verbose=self.verbose)
    self.tracker.epoch_start()
    return self

  def __exit__(self, exc_type, exc, tb):
    self.tracker.epoch_end()
    self.tracker.stop()
    self._stdout_ctx.__exit__(exc_type, exc, tb)
    output = self.buffer.getvalue()

    match_co2 = re.search(r"CO2eq:\s*([\d\.]+)\s*g", output)
    self.co2_g = float(match_co2.group(1)) if match_co2 else None

    match_energy = re.search(r"Energy:\s*([\d\.]+)\s*kWh", output)
    self.energy_kwh = float(match_energy.group(1)) if match_energy else None

# 📊 3. Carregamento dos dados

In [ ]:
ts = ls.read_file(CONFIG["dataset"])
freq = ts.index.freqstr
print(f"✅ Total de períodos: {len(ts)}")
print(f"📅 Frequência: {freq}")

X, y = ts.split(start=CONFIG["train_start"], end=CONFIG["train_end"], periods=CONFIG["forecast_horizon"])
print(f"✅ Total de períodos de treino: {len(X)}")
print(f"✅ Total de períodos de validação: {len(y)}")

model = ls.LLM(CONFIG["model"], CONFIG["api_key"], CONFIG["base_url"])
print(f"✅ Modelo: {CONFIG["model"]}")

✅ Total de períodos: 17420
📅 Frequência: h
✅ Total de períodos de treino: 1680
✅ Total de períodos de validação: 7
✅ Modelo: gemma-3-27b-it


# 💬 4. Geração de prompts

In [ ]:
prompts_info = []
def add_prompt(prompt_type, tsformat, tstype, examples=0, sampling=None):
  prompts_info.append({
    "Tipo de Prompt": prompt_type,
    "Formato": tsformat,
    "Tipo de Série": tstype,
    "Exemplos": examples,
    "Estratégia de Amostragem": sampling,
    "Prompt": ls.prompt(type=prompt_type, ts=X, tsformat=tsformat, tstype=tstype, forecast_horizon=CONFIG["forecast_horizon"],
                        examples=examples, sampling=sampling, stl=X.stl(freq=CONFIG["freq"]))})

for prompt_type in ["zero_shot", "cot"]:
  for tsformat in CONFIG["formats"]:
    for tstype in CONFIG["types"]:
      add_prompt(prompt_type, tsformat, tstype)

for prompt_type in ["few_shot", "cot_few"]:
  for sampling in CONFIG["sampling"]:
    for examples in CONFIG["top_k"]:
      for tsformat in CONFIG["formats"]:
        for tstype in CONFIG["types"]:
          add_prompt(prompt_type, tsformat, tstype, examples, sampling)

print(f"✅ Total de prompts: {len(prompts_info)}")

✅ Total de prompts: 16


In [ ]:
print(prompts_info[10]["Prompt"].text)

[OBJECTIVE]
Predict the next 7 values based on the historical series (1680 periods).

[STATISTICAL CONTEXT]
- Mean: 41.3191
- Median: 41.616
- Standard Deviation: 6.7386
- Minimum Value: 24.258
- Maximum Value: 58.438
- First Quartile (Q1): 36.562
- Third Quartile (Q3): 46.011
- Trend Strength (STL): 0.9425
- Seasonality Strength (STL): 0.8806

[EXAMPLES]
- Example 1:
Input (history):
date,value
2016-07-12 06:00:00,37.124
2016-07-12 07:00:00,36.904
2016-07-12 08:00:00,37.563
2016-07-12 09:00:00,39.541
2016-07-12 10:00:00,41.738
2016-07-12 11:00:00,44.155
2016-07-12 12:00:00,46.572

Output (forecast):
date,value
2016-07-12 13:00:00,47.012
2016-07-12 14:00:00,49.429
2016-07-12 15:00:00,49.209
2016-07-12 16:00:00,47.451
2016-07-12 17:00:00,47.012
2016-07-12 18:00:00,46.353
2016-07-12 19:00:00,45.254

- Example 2:
Input (history):
date,value
2016-08-15 23:00:00,45.791
2016-08-16 00:00:00,45.132
2016-08-16 01:00:00,44.473
2016-08-16 02:00:00,43.594
2016-08-16 03:00:00,42.935
2016-08-16 04:0

# 🧪 5. Geração de experimentos

In [ ]:
experiments = [
    (temp, prompt_info, run_id)
    for temp in CONFIG["temperatures"]
    for prompt_info in prompts_info
    for run_id in range(CONFIG["runs_per_prompt"])
]
total_experiments = len(experiments)
print(f"✅ Total de experimentos: {total_experiments}")

✅ Total de experimentos: 80


# 🚀 6. Execução dos experimentos

In [ ]:
results = []
progress = 0
total_errors = 0
total_inferences = 0

In [ ]:
consecutive_errors = 0
exec_start_time = time.perf_counter()

while progress < total_experiments:
  temperature, prompt_info, run_id = experiments[progress]

  if consecutive_errors >= CONFIG["max_retries"]:
    print(f"❌ Máximo de erros consecutivos ({CONFIG["max_retries"]}). Abortando.")
    break

  try:
    eco2ai_tracker.start()
    with CarbonTracker() as carbontracker:
      response = model.predict(prompt_info["Prompt"], temperature=temperature)

    # print(response.raw)
    total_inferences += 1

    carbontracker_g = carbontracker.co2_g
    carbontracker_kWh = carbontracker.energy_kwh
    carbontracker_Wh = carbontracker_kWh * 1000

    eco2ai_tracker.stop()
    eco2ai_data = pd.read_csv("eco2ai_emissions.csv").iloc[-1]
    eco2ai_kWh = eco2ai_data["power_consumption(kWh)"]
    eco2ai_Wh = eco2ai_kWh * 1000
    eco2ai_kg = eco2ai_data["CO2_emissions(kg)"]
    eco2ai_g = eco2ai_kg * 1000

    try:
      prediction = response.prediction
      metrics = ls.metrics(y, prediction)["value"]
    except Exception:
        raise

    input_cost = (response.input_tokens / 1_000_000) * CONFIG["input_price"]
    output_cost = (response.output_tokens / 1_000_000) * CONFIG["output_price"]
    total_cost = input_cost + output_cost

    results.append({
        "Dataset": CONFIG["dataset"],
        "Início do Treinamento": CONFIG["train_start"],
        "Fim do Treinamento": CONFIG["train_start"],
        "Horizonte de Previsão": CONFIG["forecast_horizon"],
        "Modelo": CONFIG["model"],
        "Temperatura": temperature,
        "Tipo de Série": prompt_info["Tipo de Série"],
        "Tipo de Prompt": prompt_info["Tipo de Prompt"],
        "Exemplos": prompt_info["Exemplos"],
        "Estratégia de Amostragem": prompt_info["Estratégia de Amostragem"],
        "Formato": prompt_info["Formato"],
        "Horizonte de Previsão": CONFIG["forecast_horizon"],
        "sMAPE": round(metrics["smape"], 4),
        "MAE": round(metrics["mae"], 4),
        "RMSE": round(metrics["rmse"], 4),
        "Custo Financeiro ($)": round(total_cost, 6),
        "CarbonTracker CO₂ (g)": round(carbontracker_g, 6),
        "CarbonTracker Consumo (Wh)": round(carbontracker_Wh, 6),
        "Eco2AI CO₂ (g)": round(eco2ai_g, 6),
        "Eco2AI Consumo (Wh)": round(eco2ai_Wh, 6),
        "Tokens de Entrada": response.input_tokens,
        "Tokens de Saída": response.output_tokens,
        "Tempo de Inferência (s)": response.time,
        "Valores Reais": list(y),
        "Valores Preditos": list(prediction),
        "Prompt": prompt_info["Prompt"].text
    })

    filled = int((run_id + 1) / CONFIG["runs_per_prompt"] * CONFIG["runs_per_prompt"])
    bar = "■" * filled + "□" * (CONFIG["runs_per_prompt"] - filled)

    status = f"✅ [{progress+1}/{total_experiments}] "
    status += f"[{bar}] {run_id + 1}/{CONFIG["runs_per_prompt"]} | "
    status += f"{prompt_info["Tipo de Prompt"]} | "
    status += f"{prompt_info["Formato"]} | "
    status += f"{prompt_info["Tipo de Série"]} | "
    status += f"sMAPE: {metrics["smape"]:.3f} | "
    status += f"MAE: {metrics["mae"]:.3f} | "
    status += f"RMSE: {metrics["rmse"]:.3f} | "
    status += f"CO₂: {carbontracker_g:.6f} g | "
    status += f"Custo: ${total_cost:.6f} | "
    status += f"({response.time:.2f}s)"
    print(status)
    progress += 1
    consecutive_errors = 0
    time.sleep(CONFIG["delay"])

  except Exception as e:
    total_errors += 1
    consecutive_errors += 1
    print(f"⚠️ Erro ({consecutive_errors}/{CONFIG["max_retries"]}): {str(e)[:120]}")
    time.sleep(CONFIG["delay"])

exec_end_time = time.perf_counter()
total_exec_time = exec_end_time - exec_start_time
accuracy = 1 - (total_errors / total_inferences) if total_inferences else 0.0

print(f"\n🎉 Finalizado: {progress}/{total_experiments} experimentos")
print(f"✔️ Inferências totais: {total_inferences}")
print(f"❌ Erros totais: {total_errors}")
print(f"🎯 Acurácia: {accuracy:.2%}")
print(f"⏱️ Tempo total de execução: {total_exec_time:.2f}s")

[INFO] Requested http://ipinfo.io/json
[INFO] HTTP Request: POST http://192.168.68.113:1234/v1/chat/completions "HTTP/1.1 200 OK"


✅ [2/80] [■■□□□] 2/5 | zero_shot | plain | numeric | sMAPE: 2.660 | MAE: 0.800 | RMSE: 0.880 | CO₂: 0.155275 g | Custo: $0.000000 | (8.87s)


[INFO] Requested http://ipinfo.io/json
[INFO] HTTP Request: POST http://192.168.68.113:1234/v1/chat/completions "HTTP/1.1 200 OK"


✅ [3/80] [■■■□□] 3/5 | zero_shot | plain | numeric | sMAPE: 10.570 | MAE: 3.420 | RMSE: 4.480 | CO₂: 0.157010 g | Custo: $0.000000 | (8.86s)


[INFO] Requested http://ipinfo.io/json
[INFO] HTTP Request: POST http://192.168.68.113:1234/v1/chat/completions "HTTP/1.1 200 OK"


✅ [4/80] [■■■■□] 4/5 | zero_shot | plain | numeric | sMAPE: 14.860 | MAE: 4.650 | RMSE: 5.970 | CO₂: 0.157981 g | Custo: $0.000000 | (8.88s)


[INFO] Requested http://ipinfo.io/json
[INFO] HTTP Request: POST http://192.168.68.113:1234/v1/chat/completions "HTTP/1.1 200 OK"


✅ [5/80] [■■■■■] 5/5 | zero_shot | plain | numeric | sMAPE: 2.630 | MAE: 0.790 | RMSE: 0.840 | CO₂: 0.164046 g | Custo: $0.000000 | (8.87s)


[INFO] Requested http://ipinfo.io/json


# 💾 7. Salvar resultados

In [ ]:
df = pd.DataFrame(results)
os.makedirs(CONFIG["folder"], exist_ok=True)
df.to_csv(f"{CONFIG["folder"]}/resultado_experimentos.csv", index=False)
print(f"✅ Resultados salvos: {len(df)} experimentos")
df

✅ Resultados salvos: 80 experimentos


,Dataset,Início do Treinamento,Fim do Treinamento,Horizonte de Previsão,Modelo,Temperatura,Tipo de Série,Tipo de Prompt,Exemplos,Estratégia de Amostragem,...,CarbonTracker CO₂ (g),CarbonTracker Consumo (Wh),Eco2AI CO₂ (g),Eco2AI Consumo (Wh),Tokens de Entrada,Tokens de Saída,Tempo de Inferência (s),Valores Reais,Valores Preditos,Prompt
0,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,numeric,zero_shot,0,None,...,0.857995,8.312914,0.584134,5.166176,55510,234,37.868020,"[25.35650062561035, 25.136999130249023, 27.333...","[25.357, 24.917, 25.357, 27.334, 28.652, 30.19...",[OBJECTIVE]\nPredict the next 7 values based o...
1,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,numeric,zero_shot,0,None,...,0.163712,1.586164,0.116193,1.027630,55510,234,8.874043,"[25.35650062561035, 25.136999130249023, 27.333...","[25.796, 26.455, 28.332, 31.289, 34.585, 37.44...",[OBJECTIVE]\nPredict the next 7 values based o...
2,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,numeric,zero_shot,0,None,...,0.162186,1.571381,0.120589,1.066506,55510,234,8.841393,"[25.35650062561035, 25.136999130249023, 27.333...","[25.796, 26.455, 28.332, 31.289, 34.365, 37.44...",[OBJECTIVE]\nPredict the next 7 values based o...
3,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,numeric,zero_shot,0,None,...,0.163597,1.585056,0.120961,1.069798,55510,234,8.874217,"[25.35650062561035, 25.136999130249023, 27.333...","[25.796, 25.357, 24.917, 25.357, 25.796, 26.45...",[OBJECTIVE]\nPredict the next 7 values based o...
4,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,numeric,zero_shot,0,None,...,0.162873,1.578037,0.117366,1.038003,55510,234,8.858477,"[25.35650062561035, 25.136999130249023, 27.333...","[25.357, 24.917, 25.576, 28.433, 31.069, 33.70...",[OBJECTIVE]\nPredict the next 7 values based o...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,textual,cot_few,3,uniform,...,0.917381,8.888288,0.643464,5.690893,53494,213,46.453386,"[25.35650062561035, 25.136999130249023, 27.333...","[3.5, 4.1, 5.8, 7.1, 8.0, 7.5, 6.2]",[OBJECTIVE]\nPredict the next 7 values based o...
76,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,textual,cot_few,3,uniform,...,0.163786,1.586882,0.137525,1.216297,53494,227,8.450611,"[25.35650062561035, 25.136999130249023, 27.333...","[4.585, 4.366, 4.258, 5.796, 8.652, 8.979, 9.531]",[OBJECTIVE]\nPredict the next 7 values based o...
77,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,textual,cot_few,3,uniform,...,0.163597,1.585053,0.135803,1.201066,53494,227,8.464068,"[25.35650062561035, 25.136999130249023, 27.333...","[4.805, 4.917, 6.236, 7.773, 8.334, 8.433, 8.213]",[OBJECTIVE]\nPredict the next 7 values based o...
78,data/ETTh2.csv,2016-07-12 06:00:00,2016-07-12 06:00:00,7,gemma-3-27b-it,0.7,textual,cot_few,3,uniform,...,0.156577,1.517039,0.131742,1.165151,53494,220,8.191694,"[25.35650062561035, 25.136999130249023, 27.333...","[3.78, 4.56, 5.44, 6.45, 7.46, 8.01, 7.66]",[OBJECTIVE]\nPredict the next 7 values based o...


# 💾 8. Salvar resumo dos resultados

In [ ]:
df = pd.read_csv(f"{CONFIG["folder"]}/resultado_experimentos.csv")

numeric_cols = [
    "sMAPE", "MAE", "RMSE",
    "Tempo de Inferência (s)", "Custo Financeiro ($)",
    "CarbonTracker CO₂ (g)", "CarbonTracker Consumo (Wh)",
    "Eco2AI CO₂ (g)", "Eco2AI Consumo (Wh)",
    "Tokens de Entrada", "Tokens de Saída"
]

cols_categoricas = df.columns.difference(numeric_cols).tolist()

# SEM
df_sem = (
    df.groupby("Prompt")["sMAPE"]
      .apply(lambda x: sem(x, nan_policy="omit") if len(x) > 1 else 0.0)
      .reset_index(name="SEM")
)

# Média das colunas numéricas
df_mean = df.groupby("Prompt", as_index=False)[numeric_cols].mean()
df_mean.rename(columns={col: f"{col} Média" for col in numeric_cols}, inplace=True)

# Primeira ocorrência das colunas categóricas
df_first = (
    df.groupby("Prompt", as_index=False)[cols_categoricas]
      .first()
)

# Merge final
df_mean = (
    df_mean
    .merge(df_first, on="Prompt", how="left")
    .merge(df_sem, on="Prompt", how="left")
)

df_mean.to_csv(f"{CONFIG["folder"]}/experimentos_resumo.csv", index=False)
print(f"✅ Total de experimentos únicos: {len(df_mean)} experimentos")
df_mean

✅ Total de experimentos únicos: 10 experimentos


,Prompt,sMAPE Média,MAE Média,RMSE Média,Tempo de Inferência (s) Média,Custo Financeiro ($) Média,CarbonTracker CO₂ (g) Média,CarbonTracker Consumo (Wh) Média,Eco2AI CO₂ (g) Média,Eco2AI Consumo (Wh) Média,...,Formato,Horizonte de Previsão,Início do Treinamento,Modelo,Temperatura,Tipo de Prompt,Tipo de Série,Valores Preditos,Valores Reais,SEM
0,[OBJECTIVE]\nPredict the next 7 values based o...,140.6720,25.552,25.922,16.108704,0.0,0.314460,3.046736,0.230505,2.038617,...,csv,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,few_shot,textual,"[3.852, 4.067, 4.813, 6.254, 7.556, 8.645, 9.122]","[25.35650062561035, 25.136999130249023, 27.333...",6.878481
1,[OBJECTIVE]\nPredict the next 7 values based o...,10.6600,3.374,4.428,16.405159,0.0,0.286440,2.775245,0.251973,2.228488,...,csv,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,few_shot,numeric,"[25.796, 25.576, 26.236, 28.433, 30.63, 32.388...","[25.35650062561035, 25.136999130249023, 27.333...",2.612811
2,[OBJECTIVE]\nPredict the next 7 values based o...,9.2000,2.982,4.002,19.979618,0.0,0.402306,3.897848,0.283470,2.507052,...,plain,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,few_shot,textual,"[25.796, 25.796, 26.455, 28.32, 30.191, 31.396...","[25.35650062561035, 25.136999130249023, 27.333...",1.582514
3,[OBJECTIVE]\nPredict the next 7 values based o...,9.4560,2.980,3.752,16.967189,0.0,0.334547,3.241349,0.262662,2.323025,...,plain,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,few_shot,numeric,"[25.796, 26.455, 28.213, 30.63, 32.607, 35.025...","[25.35650062561035, 25.136999130249023, 27.333...",2.584509
4,[OBJECTIVE]\nPredict the next 7 values based o...,49.8045,9.437,10.928,16.092240,0.0,0.308643,2.990370,0.234300,2.072183,...,plain,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,zero_shot,numeric,"[25.357, 24.917, 25.357, 27.334, 28.652, 30.19...","[25.35650062561035, 25.136999130249023, 27.333...",14.856158
5,[OBJECTIVE]\nPredict the next 7 values based o...,132.7180,24.694,24.990,16.000648,0.0,0.313030,3.032876,0.236699,2.093408,...,csv,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,cot_few,textual,"[3.5, 4.1, 5.8, 7.1, 8.0, 7.5, 6.2]","[25.35650062561035, 25.136999130249023, 27.333...",1.497823
6,[OBJECTIVE]\nPredict the next 7 values based o...,14.3240,4.378,5.514,18.498887,0.0,0.296001,2.867888,0.219355,1.940005,...,csv,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,cot_few,numeric,"[25.796, 25.357, 24.917, 25.357, 26.236, 27.11...","[25.35650062561035, 25.136999130249023, 27.333...",3.124735
7,[OBJECTIVE]\nPredict the next 7 values based o...,6.9740,2.266,3.010,19.730919,0.0,0.394837,3.825484,0.277370,2.453101,...,plain,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,cot_few,textual,"[25.796, 25.796, 26.236, 27.442, 28.652, 29.41...","[25.35650062561035, 25.136999130249023, 27.333...",1.758328
8,[OBJECTIVE]\nPredict the next 7 values based o...,9.1280,2.902,3.692,17.112441,0.0,0.336785,3.263037,0.262133,2.318345,...,plain,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,cot_few,numeric,"[25.796, 25.357, 25.357, 26.455, 28.213, 30.19...","[25.35650062561035, 25.136999130249023, 27.333...",2.175381
9,[OBJECTIVE]\nPredict the next 7 values based o...,47.1310,8.727,9.646,16.903197,0.0,0.322218,3.121891,0.240558,2.127535,...,plain,7,2016-07-12 06:00:00,gemma-3-27b-it,0.7,cot,numeric,"[25.796, 26.236, 28.433, 30.63, 32.827, 35.244...","[25.35650062561035, 25.136999130249023, 27.333...",15.412366


# 📊 9. Comparação entre formatos por sMAPE médio

In [ ]:
fig = px.bar(
  df.groupby("Formato", as_index=False)["sMAPE"].mean().sort_values("sMAPE"),
  x="Formato", y="sMAPE",
  title="Comparação entre formatos por sMAPE médio",
  labels={"sMAPE": "sMAPE médio"},
  color="Formato"
)
fig.update_layout(showlegend=False, yaxis_tickformat=".2f", title_x=0.5)
fig

# 📊 10. Comparação entre prompts por sMAPE médio

In [ ]:
fig = px.bar(
    df.groupby("Tipo de Prompt", as_index=False)["sMAPE"].mean().sort_values("sMAPE"),
    x="Tipo de Prompt", y="sMAPE",
    color="Tipo de Prompt",
    title="Comparação entre prompts por sMAPE médio",
    labels={"Tipo de Prompt": "Prompt", "sMAPE": "sMAPE médio"}
)
fig.update_layout(showlegend=True, yaxis_tickformat=".2f", title_x=0.5)
fig

# 📊 11. Comparação entre tipos de série por sMAPE médio

In [ ]:
fig = px.bar(
    df.groupby("Tipo de Série", as_index=False)["sMAPE"].mean().sort_values("sMAPE"),
    x="Tipo de Série", y="sMAPE",
    color="Tipo de Série",
    title="Comparação entre tipos de série por sMAPE médio",
    labels={"sMAPE": "sMAPE médio"}
)
fig.update_layout(showlegend=True, yaxis_tickformat=".2f", title_x=0.5)
fig

# 📈 12. Gráfico Real x Predito

In [ ]:
df = pd.read_csv(f"{CONFIG["folder"]}/resultado_experimentos.csv")

row = df.iloc[30]
prompt = row["Tipo de Prompt"].upper()
formato = row["Formato"].upper()
tipo = row["Tipo de Série"].upper()
smape = row["sMAPE"]
mae = row["MAE"]
rmse = row["RMSE"]

ts_val = ls.UniTimeSeries(eval(row["Valores Reais"]))
ts_pred = ls.UniTimeSeries(eval(row["Valores Preditos"]))

ls.lineplot(ts_val, ts_pred, title=f"{prompt} / {formato} / {tipo} / sMAPE = {smape} / MAE = {mae} / RMSE = {rmse}", groups=["Real", "Predicted"])